In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from pathlib import Path
from matplotlib.lines import Line2D


In [ ]:
def plot_phenotype_association_heatmap(
    phenotype_files,
    trait_name_map=None,
    phenotype_name_map=None,
    *,
    trait_col='trait',
    phenotype_col='phenotype',
    variance_col='partial_r2',
    marginal_fdr_col='univariate_fdr',
    conditional_fdr_col='multivariate_fdr',
    signal_col='independent_signal',
    fdr_threshold=0.05,
    require_conditional_also_marginal=False,
    only_show_phenotypes_with_conditional=False,
    figsize=(12, 7),
    dpi=120,
    cmap='Reds',
    vmin=0.0,
    vmax=None,
    cbar_label='Variance explained (%)',
    title='Phenotype associations',
    title_fontsize=14,
    axis_label_fontsize=12,
    tick_fontsize=10,
    colorbar_tick_fontsize=10,
    colorbar_label_fontsize=11,
    marginal_star_size=130,
    conditional_star_size=95,
    signal_fontsize=9,
    x_rotation=45,
    y_rotation=0,
):
    trait_name_map = trait_name_map or {}
    phenotype_name_map = phenotype_name_map or {}

    frames = []
    for f in phenotype_files:
        f = Path(f)
        df = pd.read_csv(f, sep='	')
        if 'Unnamed: 0' in df.columns and trait_col not in df.columns:
            df = df.drop(columns=['Unnamed: 0'])
        frames.append(df)

    if not frames:
        raise ValueError('No phenotype files provided.')

    data = pd.concat(frames, ignore_index=True)

    required = [trait_col, phenotype_col, signal_col, variance_col, marginal_fdr_col, conditional_fdr_col]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}. Available columns: {list(data.columns)}')

    data = data.copy()
    data[variance_col] = pd.to_numeric(data[variance_col], errors='coerce')
    data[marginal_fdr_col] = pd.to_numeric(data[marginal_fdr_col], errors='coerce')
    data[conditional_fdr_col] = pd.to_numeric(data[conditional_fdr_col], errors='coerce')
    data[signal_col] = pd.to_numeric(data[signal_col], errors='coerce').fillna(-1).astype(int)

    # Convert fractional R2 to percent if needed.
    if data[variance_col].max(skipna=True) <= 1.5:
        data[variance_col] = data[variance_col] * 100.0

    data['trait_display'] = data[trait_col].map(lambda t: trait_name_map.get(t, t))
    data['phenotype_display'] = data[phenotype_col].map(lambda p: phenotype_name_map.get(p, p))

    data['is_marginal_sig'] = data[marginal_fdr_col] <= fdr_threshold
    data['is_conditional_sig'] = data[conditional_fdr_col] <= fdr_threshold
    if require_conditional_also_marginal:
        data['is_conditional_sig'] = data['is_conditional_sig'] & data['is_marginal_sig']

    if only_show_phenotypes_with_conditional:
        keep = data.groupby('phenotype_display')['is_conditional_sig'].any()
        keep = keep[keep].index
        data = data[data['phenotype_display'].isin(keep)]

    if data.empty:
        raise ValueError('No rows left after filtering.')

    trait_order = list(dict.fromkeys(data['trait_display'].tolist()))
    pheno_order = list(dict.fromkeys(data['phenotype_display'].tolist()))

    # Traits on Y-axis, phenotypes on X-axis
    variance_mat = data.pivot_table(index='trait_display', columns='phenotype_display', values=variance_col, aggfunc='mean')
    marginal_mat = data.pivot_table(index='trait_display', columns='phenotype_display', values='is_marginal_sig', aggfunc='max').fillna(False)
    conditional_mat = data.pivot_table(index='trait_display', columns='phenotype_display', values='is_conditional_sig', aggfunc='max').fillna(False)
    signal_mat = data.pivot_table(index='trait_display', columns='phenotype_display', values=signal_col, aggfunc='first').fillna(-1).astype(int)

    variance_mat = variance_mat.reindex(index=trait_order, columns=pheno_order)
    marginal_mat = marginal_mat.reindex(index=trait_order, columns=pheno_order).fillna(False)
    conditional_mat = conditional_mat.reindex(index=trait_order, columns=pheno_order).fillna(False)
    signal_mat = signal_mat.reindex(index=trait_order, columns=pheno_order).fillna(-1).astype(int)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    hm = sns.heatmap(
        variance_mat,
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'label': cbar_label},
    )

    ax.set_title('')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=x_rotation, ha='right', fontsize=tick_fontsize)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=y_rotation, fontsize=tick_fontsize)

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=colorbar_tick_fontsize)
    cbar.set_label(cbar_label, size=colorbar_label_fontsize)

    n_rows, n_cols = variance_mat.shape
    for i in range(n_rows):
        for j in range(n_cols):
            is_conditional = bool(conditional_mat.iloc[i, j])
            is_marginal = bool(marginal_mat.iloc[i, j])
            if is_marginal and not is_conditional:
                ax.scatter(j + 0.5, i + 0.5, marker='*', s=marginal_star_size,
                           facecolors='none', edgecolors='black', linewidths=1.2, zorder=5)
            if is_conditional:
                ax.scatter(j + 0.5, i + 0.5, marker='*', s=conditional_star_size,
                           facecolors='lightskyblue', edgecolors='black', linewidths=1.0, zorder=6)

    # Renumber independent signals per trait row from left to right (1,2,3,...)
    # using only conditionally-associated cells.
    for i, trait in enumerate(trait_order):
        cond_row = conditional_mat.loc[trait]
        row_signal_vals = signal_mat.loc[trait]

        cond_indices = [j for j in range(len(pheno_order)) if bool(cond_row.iloc[j])]
        conditional_signal_ids = [
            int(row_signal_vals.iloc[j])
            for j in cond_indices
            if int(row_signal_vals.iloc[j]) >= 0
        ]
        ordered_unique_ids = list(dict.fromkeys(conditional_signal_ids))

        # Fallback: if conditional associations exist but all independent_signal are -1,
        # lump all conditionally-associated phenotypes into one signal labeled "1".
        fallback_one_signal = len(cond_indices) > 0 and len(ordered_unique_ids) == 0
        signal_num_map = {sid: k + 1 for k, sid in enumerate(ordered_unique_ids)}

        for j, pheno in enumerate(pheno_order):
            if not bool(cond_row.iloc[j]):
                continue
            sid = int(signal_mat.loc[trait, pheno])
            if fallback_one_signal:
                label_text = '1'
            elif sid < 0 or sid not in signal_num_map:
                continue
            else:
                label_text = str(signal_num_map[sid])

            txt = ax.text(
                j + 0.93,
                i + 0.1,
                label_text,
                color='lightskyblue',
                fontsize=signal_fontsize,
                ha='right',
                va='top',
                zorder=7,
            )
            txt.set_path_effects([pe.Stroke(linewidth=1.8, foreground='black'), pe.Normal()])


    legend_handles = [
        Line2D([0], [0], marker='*', linestyle='None', markerfacecolor='none', markeredgecolor='black',
               markeredgewidth=1.2, markersize=10, label='Sig. marginal assoc.'),
        Line2D([0], [0], marker='*', linestyle='None', markerfacecolor='lightskyblue', markeredgecolor='black',
               markeredgewidth=1.0, markersize=10, label='Sig. fine-mapped assoc.'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1.02), frameon=True)

    plotted_data = {
        'raw': data,
        'variance_mat': variance_mat,
        'marginal_sig_mat': marginal_mat,
        'conditional_sig_mat': conditional_mat,
        'signal_mat': signal_mat,
        'used_columns': {
            'variance_col': variance_col,
            'marginal_fdr_col': marginal_fdr_col,
            'conditional_fdr_col': conditional_fdr_col,
        },
    }
    return fig, ax, plotted_data


In [ ]:
import matplotlib.pyplot as plt

results_dir = "ct_validations/soskic/scdrs_fm_results"

gs = [
    "PASS_CD_deLange2017",
    "PASS_Celiac",
    "PASS_IBD_deLange2017",
    "PASS_Lupus",
    "PASS_Multiple_sclerosis",
    "PASS_Primary_biliary_cirrhosis",
    "PASS_Rheumatoid_Arthritis",
    "PASS_Type_1_Diabetes",
    "PASS_UC_deLange2017",
    "UKB_460K.disease_AID_ALL",
    "UKB_460K.body_HEIGHTz",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP",
    "UKB_460K.disease_RESPIRATORY_ENT",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED",
    "UKB_460K.disease_ASTHMA_DIAGNOSED",
]

trait_name_map = {
    "PASS_CD_deLange2017": "Crohn’s Disease (CD)",
    "PASS_UC_deLange2017": "Ulcerative Colitis (UC)",
    "PASS_IBD_deLange2017": "Inflammatory Bowel Disease (IBD)",
    "PASS_Celiac": "Celiac Disease (CeD)",
    "PASS_Lupus": "Lupus",
    "PASS_Multiple_sclerosis": "Multiple Sclerosis (MS)",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid Arthritis (RA)",
    "PASS_Type_1_Diabetes": "Type 1 Diabetes (T1D)",
    "PASS_Primary_biliary_cirrhosis": "Primary Biliary Cholangitis (PBC)",
    "UKB_460K.disease_AID_ALL": "Any Autoimmune Disease (AID)",
    "UKB_460K.disease_ASTHMA_DIAGNOSED": "Asthma (ASTH)",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED": "Allergy or Eczema (ALL/EZ)",
    "UKB_460K.disease_RESPIRATORY_ENT": "Respiratory / ENT Disease (RESP/ENT)",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP": "Hypothyroidism (HYPO)",
    "UKB_460K.body_HEIGHTz": "Height (z)",
}

phenotype_name_map = {}

phenotype_files = [f"{results_dir}/{trait}.marg-marg.decomposition.tsv.gz" for trait in gs]

fig, ax, plotted_data = plot_phenotype_association_heatmap(
    phenotype_files=phenotype_files,
    trait_name_map=trait_name_map,
    phenotype_name_map=phenotype_name_map,
    require_conditional_also_marginal=False,
    only_show_phenotypes_with_conditional=False,
    figsize=(14, 8),
    title="Soskic scDRS-FM phenotype associations: marg-marg",
)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

results_dir = "ct_validations/soskic/scdrs_fm_results"

gs = [
    "PASS_CD_deLange2017",
    "PASS_Celiac",
    "PASS_IBD_deLange2017",
    "PASS_Lupus",
    "PASS_Multiple_sclerosis",
    "PASS_Primary_biliary_cirrhosis",
    "PASS_Rheumatoid_Arthritis",
    "PASS_Type_1_Diabetes",
    "PASS_UC_deLange2017",
    "UKB_460K.disease_AID_ALL",
    "UKB_460K.body_HEIGHTz",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP",
    "UKB_460K.disease_RESPIRATORY_ENT",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED",
    "UKB_460K.disease_ASTHMA_DIAGNOSED",
]

trait_name_map = {
    "PASS_CD_deLange2017": "Crohn’s Disease (CD)",
    "PASS_UC_deLange2017": "Ulcerative Colitis (UC)",
    "PASS_IBD_deLange2017": "Inflammatory Bowel Disease (IBD)",
    "PASS_Celiac": "Celiac Disease (CeD)",
    "PASS_Lupus": "Lupus",
    "PASS_Multiple_sclerosis": "Multiple Sclerosis (MS)",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid Arthritis (RA)",
    "PASS_Type_1_Diabetes": "Type 1 Diabetes (T1D)",
    "PASS_Primary_biliary_cirrhosis": "Primary Biliary Cholangitis (PBC)",
    "UKB_460K.disease_AID_ALL": "Any Autoimmune Disease (AID)",
    "UKB_460K.disease_ASTHMA_DIAGNOSED": "Asthma (ASTH)",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED": "Allergy or Eczema (ALL/EZ)",
    "UKB_460K.disease_RESPIRATORY_ENT": "Respiratory / ENT Disease (RESP/ENT)",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP": "Hypothyroidism (HYPO)",
    "UKB_460K.body_HEIGHTz": "Height (z)",
}

phenotype_name_map = {}

phenotype_files = [f"{results_dir}/{trait}.marg-cond.decomposition.tsv.gz" for trait in gs]

fig, ax, plotted_data = plot_phenotype_association_heatmap(
    phenotype_files=phenotype_files,
    trait_name_map=trait_name_map,
    phenotype_name_map=phenotype_name_map,
    require_conditional_also_marginal=False,
    only_show_phenotypes_with_conditional=False,
    figsize=(14, 8),
    title="Soskic scDRS-FM phenotype associations: marg-cond",
)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

results_dir = "ct_validations/soskic/scdrs_fm_results"

gs = [
    "PASS_CD_deLange2017",
    "PASS_Celiac",
    "PASS_IBD_deLange2017",
    "PASS_Lupus",
    "PASS_Multiple_sclerosis",
    "PASS_Primary_biliary_cirrhosis",
    "PASS_Rheumatoid_Arthritis",
    "PASS_Type_1_Diabetes",
    "PASS_UC_deLange2017",
    "UKB_460K.disease_AID_ALL",
    "UKB_460K.body_HEIGHTz",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP",
    "UKB_460K.disease_RESPIRATORY_ENT",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED",
    "UKB_460K.disease_ASTHMA_DIAGNOSED",
]

trait_name_map = {
    "PASS_CD_deLange2017": "Crohn’s Disease (CD)",
    "PASS_UC_deLange2017": "Ulcerative Colitis (UC)",
    "PASS_IBD_deLange2017": "Inflammatory Bowel Disease (IBD)",
    "PASS_Celiac": "Celiac Disease (CeD)",
    "PASS_Lupus": "Lupus",
    "PASS_Multiple_sclerosis": "Multiple Sclerosis (MS)",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid Arthritis (RA)",
    "PASS_Type_1_Diabetes": "Type 1 Diabetes (T1D)",
    "PASS_Primary_biliary_cirrhosis": "Primary Biliary Cholangitis (PBC)",
    "UKB_460K.disease_AID_ALL": "Any Autoimmune Disease (AID)",
    "UKB_460K.disease_ASTHMA_DIAGNOSED": "Asthma (ASTH)",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED": "Allergy or Eczema (ALL/EZ)",
    "UKB_460K.disease_RESPIRATORY_ENT": "Respiratory / ENT Disease (RESP/ENT)",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP": "Hypothyroidism (HYPO)",
    "UKB_460K.body_HEIGHTz": "Height (z)",
}

phenotype_name_map = {}

phenotype_files = [f"{results_dir}/{trait}.cond-cond.decomposition.tsv.gz" for trait in gs]

fig, ax, plotted_data = plot_phenotype_association_heatmap(
    phenotype_files=phenotype_files,
    trait_name_map=trait_name_map,
    phenotype_name_map=phenotype_name_map,
    require_conditional_also_marginal=False,
    only_show_phenotypes_with_conditional=False,
    figsize=(14, 8),
    title="Soskic scDRS-FM phenotype associations: cond-cond",
)
plt.show()
